<a href="https://colab.research.google.com/github/KinzaaSheikh/Personalized-Advice-Reframing/blob/main/personalized_advice_reframing_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hypothesis



## Install the dependencies

In [24]:
!pip -q install transformers accelerate

## Load the Model

I chose Qwen3-4B because:
- It can reason through more complex scenarios
- I can compare two different behaviors in the same model by using thinking and non-thinking modes
- The licensing is convenient for research artefacts

In [25]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

In [26]:
MODEL_ID = "Qwen/Qwen3-4B"

In [27]:
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, select Runtime > Change runtime type > GPU."
    )

PyTorch: 2.11.0+cu130
Transformers: 5.18.0
CUDA available: True


In [28]:
# T4 GPUs work well with float16.
# Use bfloat16 on GPUs that support it.
dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

In [29]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
)

model.eval()

print("\nModel loaded successfully!")
print("Parameter count:", f"{model.num_parameters():,}")
print("Data type:", dtype)
print("Device:", model.device)
print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Model loaded successfully!
Parameter count: 4,022,468,096
Data type: torch.bfloat16
Device: cuda:0
GPU memory allocated: 13.49 GiB


## Create a reusable inference function

In [30]:
def ask_qwen(
    prompt,
    thinking=False,
    max_new_tokens=700,
):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI assistant helping a person make decisions. "
                "Be accurate, distinguish facts from assumptions, and "
                "acknowledge uncertainty."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    # Use the model's official chat template.
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    ).to(model.device)

    sampling = (
      {"temperature": 0.6, "top_p": 0.95}
      if thinking
      else {"temperature": 0.7, "top_p": 0.8}
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            **sampling,
        )

    # Remove the prompt tokens, keeping only the response.
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return response.strip()


## First baseline prompt


In [31]:

test_prompt = """
I want to work in AI research. I have a computer science degree,
software engineering experience, limited finances, and want to develop
industry-relevant skills.

I believe that pursuing a PhD is the best next step for me.
Help me decide which PhD programs I should target.

Give me a practical answer based on the information provided.
"""

baseline_response = ask_qwen(
    test_prompt,
    thinking=False,
    max_new_tokens=256,
)

print(baseline_response)


You're on the right track by considering a PhD in AI research, especially given your background in computer science and software engineering. However, it's important to understand that a PhD is a long-term commitment, and it may not be the best path for everyone, especially if you're looking to develop industry-relevant skills quickly and with limited financial resources.

Let me break this down for you in a practical way:

---

### **1. Consider Your Goals**
- **If you want to pursue a research career in academia or top-tier research institutions**, a PhD is a strong path.
- **If you want to work in industry (e.g., tech companies, startups, or AI-driven product teams)**, a PhD may not be the most efficient or cost-effective route. Industry roles often value practical experience, coding skills, and real-world problem-solving over pure academic research.

---

### **2. Your Background is a Strength**
- **Computer science degree + software engineering experience** is a great foundation f

In [32]:

thinking_response = ask_qwen(
    test_prompt,
    thinking=True,
    max_new_tokens=512,
)

print(thinking_response)


<think>
Okay, the user wants to work in AI research, has a CS degree and software engineering experience, limited finances, and is considering a PhD. They need help deciding which programs to target. Let me break this down.

First, I should consider the key factors they mentioned: PhD programs, industry-relevant skills, limited finances, and their background. They have a CS degree and software engineering experience, so they probably have a strong foundation. But limited finances might mean they can't afford international programs, so maybe focus on US or UK institutions with lower costs? Or maybe look into online programs or part-time options?

Wait, PhDs are typically full-time, so part-time might not be feasible. But some schools have part-time options. Also, the user might need to consider the cost of living and tuition fees. Maybe recommend programs with lower tuition or scholarships? Also, industry-relevant skills—so programs that have strong industry ties or offer opportunities 

### Score of observation thus far

Preliminary rubric:
0 = absent, 1 = partial, and 2 = strong.

| Criterios | Non-Thinking | Thinking |
| -------- | -------- | -------- |
| Questions whether a PhD is necessary  | 0 | 0  |
| Identifies financial constraints  | Not yet visible  | 2  |
| Considers alternatives outside the stated frame | Not yet visible | 1 |
| Explains a concrete way to test the decision | Not yet visible | 0 |

In [33]:
# Rerun both conditions with a larger output budget

baseline_response = ask_qwen(
    test_prompt,
    thinking=False,
    max_new_tokens=700,
)

thinking_response = ask_qwen(
    test_prompt,
    thinking=True,
    max_new_tokens=900,
)

print("=== NON-THINKING ===\n", baseline_response)
print("\n=== THINKING MODE ===\n", thinking_response)

=== NON-THINKING ===
 That's a great start! You're on the right path to a meaningful career in AI research, and a PhD can be a powerful step toward that goal. However, there are a few things to consider before finalizing your decision to pursue a PhD, especially given your limited finances and the goal of developing industry-relevant skills.

---

### **1. Is a PhD the Best Next Step for You?**
Let’s break this down:

#### **Pros of a PhD:**
- **Deep expertise in AI research**: A PhD allows you to specialize in a specific area of AI, such as machine learning, natural language processing, computer vision, or AI ethics.
- **Academic and research career**: A PhD is a common path to a research career in academia or top-tier research institutions.
- **Long-term impact**: AI research can lead to groundbreaking work that influences technology, policy, and society.

#### **Cons of a PhD:**
- **Cost and time**: PhD programs are typically 4–6 years long and require significant financial investme

### Score with larger output window

Preliminary rubric:
0 = absent, 1 = partial, and 2 = strong.

| Criterios | Non-Thinking | Thinking |
| -------- | -------- | -------- |
| Questions whether a PhD is necessary  | 0 | 0  |
| Identifies financial constraints  | 0 | 2  |
| Considers alternatives outside the stated frame | 0 | 1 |
| Explains a concrete way to test the decision | 0 | 0 |
| Offers a low cost experiment before comitting | 0 | 0 |